# **Tutorial**: Camera pose from 2-D–3-D correspondences (PnP)

## What this tutorial does

**PnP** — *Perspective-n-Point* — recovers a camera's pose from a set of known
3-D points together with their observed 2-D projections.

1. Render a mesh from two camera poses with PyTorch3D, giving a **template** and
   a **target** view, each with its depth map.
2. Pick **corresponding points** between the two views by hand.
3. **Unproject** the template's picked pixels to 3-D world coordinates using its
   depth map. This supplies the 3-D half of the correspondences.
4. Solve **PnP** for the target camera's pose from those 3-D points and their
   2-D positions in the target image.
5. **Re-render the mesh from the estimated pose** and compare it against the
   original target image.

Because the scene is synthetic, the true pose is known — so the estimate can be
checked, not merely inspected.

## The problem

Given $N$ world points $\mathbf X_i$, their image projections $(u_i, v_i)$, and a
**known** intrinsic matrix $\mathbf K$, find the rotation $\Omega$ and
translation $\boldsymbol\tau$ satisfying

$$
\lambda_i\begin{bmatrix}u_i\\ v_i\\ 1\end{bmatrix}
  = \mathbf K\,[\,\Omega \mid \boldsymbol\tau\,]
    \begin{bmatrix}\mathbf X_i\\ 1\end{bmatrix}
\qquad\text{for every } i .
$$

Three correspondences are enough in principle — the minimal **P3P** case, which
returns up to four solutions and needs a fourth point to disambiguate. In
practice one uses more points and refines by minimising reprojection error.

> **How this differs from the homography route.** In the homography lecture the
> scene was a *plane*, and the pose fell out of factorizing $\Phi$. PnP makes no
> planarity assumption and works for any 3-D point set — but it requires the 3-D
> coordinates to be known in advance. Here the depth map provides them.

## Matching

Correspondences are picked **manually**, in a click-based UI. That keeps the
focus on the pose estimation itself. For automatic matching see the companion
notebook `ribeiro_feat_matching.ipynb`, which uses MASt3R.

## Before you run it

- **Use a GPU runtime.** *Runtime → Change runtime type → GPU.* PyTorch3D's
  rasterizer needs one.
- **Mount Google Drive.** PyTorch3D installs from a prebuilt wheel kept in
  `/content/drive/MyDrive/cvenv_wheels/`; if none is there, the setup cell
  builds one and saves it back for later sessions.


![link text](https://docs.opencv.org/3.4/pnp.jpg)

**Figure 1**: Points expressed in the world frame $X_w$ are projected into the image plane $[u,v]$ using the pinhole camera model (Figure from: https://docs.opencv.org/3.4/d5/d1f/calib3d_solvePnP.html).



## Mesh files


We assume that the mesh file in `.ply` (or `.obj`) format are present in the directory `assets/`.

The spaceship mesh file used in this tutorial was downloaded from: https://sketchfab.com/3d-models/spaceship-6164a883f57f4f13938c3c5999bc0e1f

In [ ]:
!pip --quiet install ipython-autotime
%load_ext autotime

## Settings and installation

Install libraries, mount Google Drive, and clone the repository.

This notebook uses [`cvenv`](https://github.com/ribeiro-computer-vision/cvenv)
for setup: one cell per component (`install` + `verify`). `pytorch3d` needs a GPU
runtime; `science` runs anywhere.

> If a step changes **numpy**, restart the runtime once before continuing.

### Install cvenv

In [ ]:
# Install cvenv itself (no heavy deps). Pin a tag for reproducibility.
!pip install -q "git+https://github.com/ribeiro-computer-vision/cvenv@v0.1.14"

import cvenv
print("cvenv", cvenv.__version__)
cvenv.PlatformManager().detect_platform()

In [ ]:
# See what's available (with the teaching notes)
for c in cvenv.list_components():
    print(f"{c.name:<10} {c.summary}")
    print(f"           why: {c.teaching_note}\n")

### science — base numpy-2 scientific stack
Runs anywhere. This is all you need for pure-numpy/scipy material.

In [ ]:
cvenv.get_component("science").install()
cvenv.get_component("science").verify()

### Mount Google Drive
On Colab, cvenv saves PyTorch3D wheels to `/content/drive/MyDrive/cvenv_wheels/`
by default, so a wheel built once is reused by later sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### pytorch3d — differentiable 3D (GPU)

Fastest from a prebuilt wheel that matches this runtime's python / torch / CUDA
and the attached GPU. If there is no usable wheel in Drive, build one with
`cvenv.get_component("pytorch3d").build_wheel(force=True)` — it is saved back to
Drive and reused by later sessions.

In [ ]:
import glob, os

wheels = glob.glob("/content/drive/MyDrive/cvenv_wheels/pytorch3d-*.whl")
if wheels:
    WHEEL = max(wheels, key=os.path.getmtime)
    print("using:", WHEEL)
    cvenv.get_component("pytorch3d").install(wheel_url=WHEEL)
else:
    print("no wheel in Drive — building one (several minutes, saved for next time)")
    WHEEL = cvenv.get_component("pytorch3d").build_wheel(force=True)
    cvenv.get_component("pytorch3d").install(wheel_url=WHEEL, force=True)

cvenv.get_component("pytorch3d").verify()   # want: ✅ _C OK + CUDA kernels run

### Colorama for printing color text

In [ ]:
import torch
import numpy as np

!pip install -q colorama
from colorama import Fore, Back, Style, init

# ---------- pretty print helpers ----------
RESET="\033[0m"; BOLD="\033[1m"
C={"ok":"\033[1;32m","info":"\033[1;36m","step":"\033[1;35m","warn":"\033[1;33m"}
CYAN  = "\033[1;36m"; GREEN = "\033[1;32m"; YELLOW = "\033[1;33m"


def say(kind,msg): print(f"{C[kind]}{msg}{RESET}")
torch.set_printoptions(precision=4, sci_mode=False)
np.set_printoptions(precision=4, suppress=True)

### Clone `https://github.com/ribeiro-computer-vision/point3D_from_depth`

The repository is public, so no token is needed.

In [ ]:
import os

repository_name = "point3D_from_depth"

if not os.path.exists(repository_name):
    !git clone "https://github.com/ribeiro-computer-vision/point3D_from_depth"

repo_name = os.path.abspath(repository_name)
print(f"✅ Repository: {repo_name}" if os.path.exists(repo_name)
      else f"❌ Repository '{repo_name}' not found. Try cloning manually.")

local_path = os.getcwd() + "/"
print("Current local path:", local_path)

### Imports

In [ ]:
#  ---------------------------- IMPORTS -----------------------------------------
# Stdlib
import os
import sys
import math
import shutil
import importlib
from pathlib import Path
from typing import Optional, Tuple, Literal, Dict, Any

# Third-party
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import imageio
import requests
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tqdm.notebook import tqdm
from skimage import img_as_ubyte

# set path for libraries
sys.path.append(repo_name)

## General settings (user input)


In [ ]:
#@title Path to mesh file (.obj)

# Path to mesh file
obj_path = "point3D_from_depth/assets/StarShip_small.obj" #@param {type:"string"}

In [ ]:
#@title 📷 Camera Intrinsics
# Focal lengths
focal_length_x = 900  #@param {type:"number"}
focal_length_y = 900  #@param {type:"number"}

# Principal point
principal_point_x = 128  #@param {type:"number"}
principal_point_y = 128  #@param {type:"number"}

# Image dimensions
image_witdh = 256   #@param {type:"number"}
image_height = 256  #@param {type:"number"}

# --- Aliases for convenience in code ---
fx, fy = focal_length_x, focal_length_y
cx, cy = principal_point_x, principal_point_y

print("\nK =")
print(f"[[{fx:8.2f} {0.0:8.2f} {cx:8.2f}]")
print(f" [{0.0:8.2f} {fy:8.2f} {cy:8.2f}]")
print(f" [{0.0:8.2f} {0.0:8.2f} {1.0:8.2f}]]\n")

# --- Aliases for convenience in code ---
W = image_witdh
H = image_height

#### PyTorch3D imports
The following cell require PyTorch3D. Ensure it is executed after PyTorch3D is installed.

In [ ]:
# # ---------------------------- IMPORTS -----------------------------------------
# PyTorch3D — IO & data structures
from pytorch3d.io import load_obj, load_ply, load_objs_as_meshes
from pytorch3d.structures import Meshes

# PyTorch3D — transforms
from pytorch3d.transforms import Rotate, Translate

# PyTorch3D — rendering
from pytorch3d.renderer import (
    FoVPerspectiveCameras,
    PerspectiveCameras,
    look_at_view_transform,
    look_at_rotation,
    camera_position_from_spherical_angles,
    RasterizationSettings,
    MeshRenderer,
    MeshRasterizer,
    BlendParams,
    SoftSilhouetteShader,
    SoftPhongShader,
    HardPhongShader,
    PointLights,
    DirectionalLights,
    Materials,
    TexturesUV,
    TexturesVertex,
)
from pytorch3d.renderer.cameras import CamerasBase

# PyTorch3D — visualization helpers (optional)
from pytorch3d.vis.plotly_vis import AxisArgs, plot_batch_individually, plot_scene
from pytorch3d.vis.texture_vis import texturesuv_image_matplotlib

# Project utils path (adjust as needed)
sys.path.append(os.path.abspath(''))
# ------------------------------------------------------------------------------


### Utility functions
These function require PyTorch3D. As a result, they must be declared after PyTorch3D is installed.

**Import my own libraries and helper functions**


In [ ]:
import unproject_3d_from_depth_tools as unproject_tools
importlib.reload(unproject_tools)

import tools_pytorch3d_coordsystems_demo as myp3dtools
importlib.reload(myp3dtools)

# estimate_pose_pnp lives here; the module no longer pulls in MASt3R at import
# time, so this works even though this notebook never installs it.
import feature_matcher_tools as featmatchtools
importlib.reload(featmatchtools)

## Read cad file into a PyTorch3D mesh

In [ ]:
# Get device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

### Read cad file

# Ensure we are in the home path
os.chdir(local_path)

# Load mesh
mesh = load_objs_as_meshes([obj_path], device=device)

## Define the camera
Ensure that this the only camera. PnP camera and rendering cameras must be the same.

In [ ]:
# Different type of matrices for different functions

# Numpy K
K_np = np.array([[fx, 0, cx],
                 [0, fy, cy],
                 [0,  0,  1]], dtype=np.float32)

# Torch K
K_torch = torch.tensor(K_np, dtype=torch.float32, device=device).unsqueeze(0)  # (1,3,3)

### Helper functions

In [ ]:
import matplotlib.pyplot as plt

def prepare_uvd_for_unproject(template_points, depth_template):

    # Get the interpolated depths for the list of (u,v) points
    points_depth = [unproject_tools.Unprojector.bilinear_sample_depth(depth_template.cpu(), uv) for uv in template_points]

    # Create the (u,v,d) to pass to unproject where d = depth
    z_cam = np.array([
      points_depth
    ], dtype=np.float32).T

    # Convert from list to np.array
    uv = np.array([
      template_points
    ], dtype=np.float32)

    # Concatenate (u,v) and depth to form (u,v,depth)
    uvd1 = np.concatenate([uv.squeeze(), z_cam], axis=1)

    # Flip image axes using image size (2-D flip, not 3-D)
    uvd1[:,0] = -uvd1[:,0] + W
    uvd1[:,1] = -uvd1[:,1] + H


    return uvd1


def recover_3D_from_depth_map(template_points, depth_template, cams_template):


    # # Create the (u,v,d) to pass to unproject where d = depth.
    # # Actually, we need to pass (W-u, H-v, cam_depth)
    uvd = prepare_uvd_for_unproject(template_points, depth_template)

    # Use Pytorch3D to unproject (u,v,depth) to camera coordinates
    # (i.e., world_coordinates = False) for our camera.
    Xcam_unproject_t = cams_template.unproject_points(torch.tensor(uvd, device="cuda:0", dtype=dtype),
                                              world_coordinates=False)

    # Also, unproject (u,v,depth) to world coordinates
    # (i.e., world_coordinates = True) for our camera.
    Xworld_unproject_t = cams_template.unproject_points(torch.tensor(uvd, device="cuda:0", dtype=dtype),
                                              world_coordinates=True)

    return Xworld_unproject_t

# Demo function to use for tests
def create_and_display_image(distance=3,
                             elev=0,
                             azim=0,
                             roll=0,
                             K=np.eye(3),
                             H=256,
                             W=256):

    # Get intrinsics
    fx = K[0,0]
    fy = K[1,1]
    cx = K[0,2]
    cy = K[1,2]

    # Create rgb and depth images
    rgb, depth, cams = unproject_tools.RenderWithPytorch3D.render_rgb_depth_from_view(
        mesh,
        fx=fx, fy=fy, cx=cx, cy=cy,
        width=W, height=H,
        distance=distance, elev=elev, azim=azim, roll_deg=roll,
        roll_mode="camera",   # try "camera" if you prefer or "world".
    )

    # Show
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1); plt.imshow(np.clip(rgb,0,1)); plt.axis('off'); plt.title('RGB')

    # Depth visualization (treat -1 as invalid)
    vis = unproject_tools.ImageProcessor.depth_to_rgb(depth, cmap="plasma", bg_mode="white")

    plt.subplot(1,2,2); plt.imshow(vis); plt.axis('off'); plt.title('Depth')
    plt.show()

    #------------------------------- Show results ---------------------------------
    myp3dtools.overlay_axes_p3d(rgb, cams, 256, 256,
                    world_origin=(0,0,0), axis_len=0.3,
                    draw_world_axes=True, draw_camera_axes=False,
                    cam_axis_len=0.5,
                    title="PyTorch3D camera")

    # Pretty print camera information
    myp3dtools.print_camera_pose_matrices(cams.R, cams.T, "*** PyTorch3D Camera ***")

    # clear gpu cache
    unproject_tools.Util.clear_cuda_cache()

    return rgb, depth, cams

import numpy as np

def add_subpixel_noise(arr, H, W, scale=0.5, scale_x=None, scale_y=None, seed=None):
    """
    Add small Gaussian subpixel noise to the first two columns (x,y) of arr.

    Args:
        arr    : (N,3) numpy array, where first 2 cols are image coords
        H, W   : image height, width (used to clip coords inside image)
        scale  : std dev of noise in pixels (applied if scale_x/scale_y not set)
        scale_x: std dev for x-axis noise (overrides scale if provided)
        scale_y: std dev for y-axis noise (overrides scale if provided)
        seed   : random seed for reproducibility (default None)

    Returns:
        arr_noisy : copy of arr with noisy first two columns
    """
    if seed is not None:
        np.random.seed(seed)

    arr_noisy = arr.copy()

    # Use separate stddev if provided
    sx = scale_x if scale_x is not None else scale
    sy = scale_y if scale_y is not None else scale

    noise_x = np.random.normal(loc=0.0, scale=sx, size=arr.shape[0])
    noise_y = np.random.normal(loc=0.0, scale=sy, size=arr.shape[0])

    arr_noisy[:, 0] += noise_x
    arr_noisy[:, 1] += noise_y

    # Clip to valid image coordinates
    arr_noisy[:, 0] = np.clip(arr_noisy[:, 0], 0, W-1)  # x
    arr_noisy[:, 1] = np.clip(arr_noisy[:, 1], 0, H-1)  # y

    return arr_noisy


In [ ]:
import numpy as np
from PIL import Image

def read_rgba_image_from_file(image_path: str, output_size: tuple[int, int]) -> np.ndarray:
    """
    Load an image as RGBA, resize it preserving aspect ratio,
    and return a (1, H, W, 4) float32 array in [0, 1].

    Args:
        image_path (str): Path to the input image.
        output_size (tuple[int, int]): Desired (width, height) of the output.

    Returns:
        np.ndarray: (1, H, W, 4) float32 array in [0, 1].
    """
    # --- 1️⃣ Load image as RGBA ---
    img = Image.open(image_path).convert("RGBA")

    # --- 2️⃣ Resize preserving aspect ratio ---
    out_w, out_h = output_size
    img.thumbnail((out_w, out_h), Image.LANCZOS)

    # --- 3️⃣ Center on transparent canvas of target size ---
    canvas = Image.new("RGBA", (out_w, out_h), (0, 0, 0, 0))
    x = (out_w - img.width) // 2
    y = (out_h - img.height) // 2
    canvas.paste(img, (x, y))

    # --- 4️⃣ Convert to numpy float32 in [0, 1] ---
    rgba = np.array(canvas).astype(np.float32) / 255.0

    # --- 5️⃣ Add batch dimension ---
    return rgba[None, ...]  # (1, H, W, 4)


# 1. 🎨 Create template RGB image and depth map

## Input image

In [ ]:
# Path to the RGBA image file
image_path = '/content/point3D_from_depth/assets/ChatGPT_Image.png'

# Reads the RGBA image and resizes it to (H, W)
# Assumes read_rgba_image_from_file returns a NumPy array (H, W, 4)
rgb_cutout_resized = read_rgba_image_from_file(image_path=image_path, output_size=(H, W))

# Remove extra dimensions if any (e.g., shape (1, H, W, 4) → (H, W, 4))
rgb_target = rgb_cutout_resized.squeeze()

# Plot the image
pl.figure()
pl.axis('off')           # Remove axes for a clean display
pl.imshow(rgb_target)    # Display the RGBA image
pl.show()

In [ ]:
#@title 🎥 Camera Pose (Template/test Image)

# Camera distance from object
distance_template = 3.9  #@param {type:"slider", min:1.0, max:10.0, step:0.1}

# Azimuth angle (horizontal rotation)
azim_template = 11  #@param {type:"slider", min:-180.0, max:180.0, step:1.0}

# Elevation angle (vertical tilt)
elev_template = 26  #@param {type:"slider", min:-90.0, max:90.0, step:1.0}

# Roll angle (rotation around camera axis)
roll_template = 0  #@param {type:"slider", min:-180.0, max:180.0, step:1.0}

print(f"Camera pose:\n  distance={distance_template}, azimuth={azim_template}, elevation={elev_template}, roll={roll_template}")



# Create a reference image
rgb_template, depth_template, cams_template = \
      create_and_display_image(distance=distance_template,
                               elev=elev_template,
                               azim=azim_template,
                               roll=roll_template,
                               K = K_np,
                               H = H,
                               W = W)



# We need these for later in the program
device = cams_template.R.device
dtype  = cams_template.R.dtype
imgsz  = torch.tensor([[H, W]], device=device)

# 2. **🧑‍💻 User Input**: Detect matching features between the template image and the input image

In [ ]:
app = unproject_tools.launch_point_matcher(rgb_template, rgb_target)

In [ ]:
from IPython import get_ipython
ip = get_ipython()

# --------------------------------------------------------------
# 🔹 Retrieve manually selected point correspondences
# --------------------------------------------------------------

# Get points selected on the template image (e.g., reference view)
template_points = ip.user_ns.get("selected_points_A", [])

# Get points selected on the target image (e.g., query view)
target_points = ip.user_ns.get("selected_points_B", [])

# --------------------------------------------------------------
# 🔹 Display the number of selected points
# --------------------------------------------------------------
print("\n")
print(f"Got {len(template_points)} points in A and {len(target_points)} points in B\n")

# --------------------------------------------------------------
# 🔹 Print table header
# --------------------------------------------------------------
print(f"{'Idx':>3} |{'Point A (u,v)':>15}  |{'Point B (u,v)':>15}")
print("-"*45)

# --------------------------------------------------------------
# 🔹 Print each pair of corresponding points
# --------------------------------------------------------------
for i, (a, b) in enumerate(zip(template_points, target_points), 0):
    # a = (u_A, v_A) in template image
    # b = (u_B, v_B) in target image
    print(f"{i:3d} | ({a[0]:4d}, {a[1]:4d})    | ({b[0]:4d}, {b[1]:4d})")

print("\n")


## Calculate depth and 3-D coordinates `(x_world, y_world, z_world)`  for the selected points `(u,v)` + depth.


Here, we use `cams.unproject(u,v,depth)` to recover the 3-D coordinates corresponding to the detected pixels. This steps gives us a set of 3-D object coordinates corresponding to the detected 2-D features.

The estimated 3-D coordinates are then back-projected on the image for visualization.

In [ ]:
# --------------------------------------------------------------
# 🔹 Recover 3D coordinates in the world (object) coordinate system
# --------------------------------------------------------------
# Given the selected 2D points in the template image (`template_points`)
# and the corresponding depth map (`depth_template`), recover their 3D
# positions in the world (object) coordinate system using camera intrinsics/extrinsics.
Xworld_unproject_t = recover_3D_from_depth_map(template_points, depth_template, cams_template)

# --------------------------------------------------------------
# 🔹 Re-project 3D points back to 2D using the same camera
# --------------------------------------------------------------
# Clone and detach the 3D tensor to avoid in-place modifications or autograd tracking.
x_world_new = Xworld_unproject_t.unsqueeze(0).detach().clone()  # Shape: (1, N, 3)

# Use PyTorch3D's transform_points_screen to map 3D world points to 2D screen (pixel) coordinates.
# The result `uvz` contains (u,v,z) for each point, where z is the depth in screen space.
uvz = cams_template.transform_points_screen(x_world_new, image_size=imgsz)[0]  # Shape: (N, 3)

# Extract only the 2D pixel coordinates (u, v)
uv_back = uvz[:, :2]

# --------------------------------------------------------------
# 🔹 Plot reprojected points for visual validation
# --------------------------------------------------------------
# Overlay the reprojected 2D points (from 3D transform) on the template image
myp3dtools.plot_re_projected_uv_on_image(
    uv_back.cpu().numpy(),  # (N, 2)
    rgb_template,           # background image
    H, W,                   # image height and width
    cams_template           # camera for plotting axes, etc.
)

# --------------------------------------------------------------
# 🔹 Console summary: compare 3D–2D correspondences
# --------------------------------------------------------------
print("\n")
print("----------------------------------------------------------------------------------------")
print("      | World(x,y,z)            | Pixel (u,v)      | Pixel (u,v)")
print("Index | from p3d unproject()    | Ground-truth     | from transform_points_screen() ")
print("----------------------------------------------------------------------------------------")

# Number of correspondences
n = len(template_points)

# Iterate over each correspondence
for i, (pt_w, (u, v), (u_back, v_back)) in enumerate(
    zip(Xworld_unproject_t, template_points, uv_back)
):

    # Highlight last few rows (optional styling)
    # color = Fore.BLUE if i >= n - 5 else ""
    color = Fore.BLACK if i >= n - 5 else ""

    # Print comparison of 3D world coordinates and their 2D projections
    print(f"{color}{i:5d} | {pt_w[0]:+7.3f} {pt_w[1]:+7.3f} {pt_w[2]:+7.3f} | "
          f"{u:7.2f} {v:7.2f}  | "
          f"{u_back:7.2f} {v_back:7.2f} {Style.RESET_ALL}")

print("----------------------------------------------------------------------------------------")





# 3. 🎯 Estimate pose pnp from 2-D-to-3-D correspondences

In [ ]:
# --------------------------------------------------------------
# 🔹 Prepare 2D–3D correspondences for PnP pose estimation
# --------------------------------------------------------------

# 2D points detected (or manually selected) in the target image
# These are the image-space pixel coordinates of matched features.
img_pts2d = target_points

# Corresponding 3D points in the object (world) coordinate system
# These come from unprojecting the template pixels via the depth map.
obj_pts3d = Xworld_unproject_t.detach().cpu()  # (M, 3)

# --------------------------------------------------------------
# 🔹 Estimate camera pose using Perspective-n-Point (PnP)
# --------------------------------------------------------------
# The goal is to find the camera rotation (R) and translation (T)
# that best align the 3D world points (obj_pts3d) with their 2D
# projections (img_pts2d) under the given camera intrinsics.
#
# - Uses OpenCV’s solvePnPRansac() internally for robust fitting
# - Optionally refines the result with Levenberg–Marquardt
# - Returns R, T in PyTorch3D-compatible form (R_p3d, T_p3d)

res = featmatchtools.estimate_pose_pnp(
    mesh,                    # 3D mesh or reference object (optional context)
    obj_pts3d,               # (M, 3) array/tensor of 3D world points
    img_pts2d,               # (M, 2) array/tensor of corresponding 2D image points
    fx, fy, cx, cy,          # Camera intrinsics (focal lengths and principal point)
    W, H,                    # Image resolution (used for normalization)
    # base_rgb=None,         # Optional background image for wireframe visualization
    # wireframe_pts3d=None,  # Optional 3D vertices for drawing wireframe
    # wireframe_edges=None,  # Optional connectivity list for wireframe plotting
    ransac=True,             # Enable RANSAC for outlier rejection
    refine=True,             # Refine final pose with nonlinear optimization
    reproj_err=2,            # Maximum reprojection error (pixels) for RANSAC inlier threshold
    iters=2000,              # Number of RANSAC iterations
    pnp_flag=None,           # Optional override (e.g., cv2.SOLVEPNP_EPNP, AP3P, ITERATIVE)
)

# --------------------------------------------------------------
# 🔹 Display numerical pose results
# --------------------------------------------------------------

# RMS reprojection error — a key quality metric for the pose
# (lower is better; typically < 2 px is excellent)
print("\nRMS reprojection error (px):", res["rms_px"])

# Rotation matrix (3×3) in PyTorch3D convention (world → camera)
print("\nRecovered R (PyTorch3D):\n", res["R_p3d"][0].cpu().numpy())

# Translation vector (3×1) in PyTorch3D convention (world → camera)
print("\nRecovered T (PyTorch3D):\n", res["T_p3d"][0].cpu().numpy())


# 4. 🖼️ Render the object using the PnP estimate

In [ ]:
# =====================================================================
# 🔹 Import required libraries
# =====================================================================
import torch, numpy as np
import cv2
from pytorch3d.renderer import PerspectiveCameras
from pytorch3d.utils import cameras_from_opencv_projection

# =====================================================================
# 🔹 Convert OpenCV PnP results into PyTorch3D camera format
# =====================================================================

# OpenCV’s PnP output (from `estimate_pose_pnp`) provides transformation
# matrices using OpenCV’s coordinate convention:
#   - R_cv, t_cv describe the world→camera transform (same as PyTorch3D).
#   - Units are consistent with the input 3D points.
R_p3d = res["R_p3d"]
T_p3d = res["T_p3d"]

# Convert to NumPy for compatibility
R_cv = R_p3d[0].detach().cpu().numpy()  # (3, 3)
t_cv = T_p3d[0].detach().cpu().numpy()  # (3,)

# ---------------------------------------------------------------------
# ✅ Create a PyTorch3D camera that exactly matches OpenCV’s projection
# ---------------------------------------------------------------------
# `cameras_from_opencv_projection` builds a PyTorch3D PerspectiveCameras
# object directly from OpenCV-style extrinsics (R, t) and intrinsics (K).
cams_pnp = cameras_from_opencv_projection(
    R=torch.tensor(R_cv, dtype=torch.float32, device=device).unsqueeze(0),        # (1,3,3)
    tvec=torch.tensor(t_cv, dtype=torch.float32, device=device).unsqueeze(0),     # (1,3)
    camera_matrix=torch.tensor(K_np, dtype=torch.float32, device=device).unsqueeze(0),  # (1,3,3)
    image_size=torch.tensor([[H, W]], dtype=torch.float32, device=device)         # (1,2)
)

# =====================================================================
# 🔹 Render the scene from the recovered PnP camera
# =====================================================================
# Generate both RGB and depth images of the mesh as seen from this camera.
rgb_target_from_pnp, depth_target_from_pnp, cams_target_from_pnp = \
    unproject_tools.RenderWithPytorch3D.render_rgb_depth_from_view_from_RT(
        mesh,
        fx=fx, fy=fy, cx=cx, cy=cy,
        width=W, height=H,
        R=cams_pnp.R,
        T=cams_pnp.T,
    )

# =====================================================================
# 🔹 Display and analyze the recovered camera
# =====================================================================

# Print rotation and translation matrices in readable format
myp3dtools.print_camera_pose_matrices(
    cams_pnp.R, cams_pnp.T, "*** PyTorch3D Camera ***"
)

# ---------------------------------------------------------------------
# 🔹 Reproject known 3D world points through the estimated camera
# ---------------------------------------------------------------------
uvz_target_pnp = cams_pnp.transform_points_screen(x_world_new, image_size=imgsz)[0]  # (N, 3)
uv_back_pnp = uvz_target_pnp[:, :2]  # Extract only pixel coordinates (u, v)

# Plot reprojected points on the rendered RGB image to visualize alignment
myp3dtools.plot_re_projected_uv_on_image(
    uv_back_pnp.cpu().numpy(), rgb_target_from_pnp, H, W, cams_pnp
)

# Free unused CUDA memory (helpful for large renders)
unproject_tools.Util.clear_cuda_cache()

# ---------------------------------------------------------------------
# ⚠️ Comparison placeholder (no ground-truth available)
# ---------------------------------------------------------------------
# This step simulates a comparison between the estimated PnP camera
# and a "reference" camera (e.g., template view). In this notebook, the
# actual object pose is unknown — the template camera is used only as a
# proxy to demonstrate how a ground-truth comparison would look.
unproject_tools.print_extrinsics_comparison_color(
    cams_template.R, cams_template.T, cams_pnp.R, cams_pnp.T
)
print("\nRMS reprojection error (px):", res["rms_px"], "\n")

# =====================================================================
# 🔹 Visualize camera axes on the rendered and real target images
# =====================================================================

# Overlay world axes on the PyTorch3D render
myp3dtools.overlay_axes_p3d(
    rgb_target_from_pnp, cams_pnp, 256, 256,
    world_origin=(0, 0, 0), axis_len=0.5,
    draw_world_axes=True, draw_camera_axes=False,
    cam_axis_len=0.5,
    title="PyTorch3D camera"
)

# Overlay the same camera axes directly on the actual target image
myp3dtools.overlay_axes_p3d(
    rgb_target, cams_pnp, 256, 256,
    world_origin=(0, 0, 0), axis_len=0.5,
    draw_world_axes=True, draw_camera_axes=False,
    cam_axis_len=0.5,
    title="PyTorch3D camera"
)
